> 在线讲义：[第 10 章 · 安全、可靠性与整机案例：怎么设计一台机器人](https://wikieden.github.io/robot-architecture/lessons/10-safety-and-cases.html) · 本书《机器人软硬件架构》

# 第 10 章 · 安全、可靠性与整机案例：怎么设计一台机器人

配套讲义：仓库里 `robot-system-architecture/notes/lessons/10-safety-and-cases.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（停止距离、力矩序列、阈值、失效概率都是编的，不是练习题的数）

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 公式表 停止距离 | $v\,t_{react} + v^2/(2a)$ | `stop_distance(0.5, 0.1, 2.0)` | 0.05 + 0.0625 = 0.1125 m |
| 第 3 步 碰撞检测 | $\lvert\tau_{meas}-\tau_{model}\rvert > \tau_{th}$ | `detect_collision` | 第 5 拍超过 2 N·m |
| 第 3 步 看门狗 | 连续几拍没心跳 | `watchdog(hb, 3)` | 第 7 拍触发 |
| 第 3 步 / 动画 状态机 | 正常 → 降速 → SS1 → STO | `step(state, event)` | 有人进入 → 降速；心跳丢失 → SS1；急停 → STO，要人工复位 |
| 数学原理 冗余 | 独立失效概率相乘 | `1e-3 * 1e-3` | 1e-6 |

停止距离用的是编的数（0.5 m/s、100 ms、2 m/s²），不是练习题 1 的数；力矩序列、阈值、失效概率也是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
NORMAL, REDUCED, SS1, STO = "正常", "降速", "保护停止 SS1", "STO"

**函数 `stop_distance`**：停止距离 d = v t_react + v² / (2a)：反应时间里匀速走，之后匀减速。

In [ ]:
def stop_distance(v: float, t_react: float, a: float) -> float:
    """停止距离 d = v t_react + v² / (2a)：反应时间里匀速走，之后匀减速。"""
    return v * t_react + v**2 / (2 * a)

**函数 `detect_collision`**：碰撞检测：第一次 |τ_meas − τ_model| > τ_th 的下标；没有返回 −1。

In [ ]:
def detect_collision(tau_meas: np.ndarray, tau_model: np.ndarray, tau_th: float) -> int:
    """碰撞检测：第一次 |τ_meas − τ_model| > τ_th 的下标；没有返回 −1。"""
    hit = np.nonzero(np.abs(tau_meas - tau_model) > tau_th)[0]
    return int(hit[0]) if hit.size else -1

**函数 `step`**：故障降级状态机（讲义第三步那张图）。

In [ ]:
def step(state: str, event: str) -> str:
    """故障降级状态机（讲义第三步那张图）。"""
    if event == "急停":
        return STO
    if state == STO:
        return NORMAL if event == "人工复位" else STO
    if event in ("心跳丢失", "力矩异常", "温度过高", "接触力超限"):
        return SS1
    if state == SS1:
        return NORMAL if event == "复位确认" else SS1
    if event == "有人进入":
        return REDUCED
    if state == REDUCED and event == "人离开":
        return NORMAL
    return state

**函数 `watchdog`**：看门狗：连续 max_missed 拍没收到心跳（0）就触发，返回触发的那一拍；不触发返回 −1。

In [ ]:
def watchdog(heartbeats: list[int], max_missed: int = 3) -> int:
    """看门狗：连续 max_missed 拍没收到心跳（0）就触发，返回触发的那一拍；不触发返回 −1。"""
    missed = 0
    for k, hb in enumerate(heartbeats):
        missed = 0 if hb else missed + 1
        if missed >= max_missed:
            return k
    return -1

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n公式表：停止距离 d = v t_react + v²/(2a)（编的：0.5 m/s、100 ms、2 m/s²）")
d = stop_distance(0.5, 0.1, 2.0)
print(f"  反应时间里走 {0.5 * 0.1:.4f} m + 减速走 {0.5**2 / 4:.4f} m = {d:.4f} m")

In [ ]:
print("\n第三步：碰撞检测 |τ_meas − τ_model| > τ_th（阈值 2 N·m）")
model = np.full(10, 5.0)
meas = model + np.array([0.1, -0.2, 0.3, 0.0, 0.2, 2.6, 4.0, 4.5, 4.2, 4.0])
print(f"  第一次超阈值在第 {detect_collision(meas, model, 2.0)} 拍（从 0 数）")

In [ ]:
print("\n第三步：看门狗——连续 3 拍没心跳就进入保护停止")
hb = [1, 1, 1, 0, 1, 0, 0, 0, 1]
print(f"  心跳 {hb} → 第 {watchdog(hb)} 拍触发")

In [ ]:
print("\n动画：一串事件走过状态机")
s = NORMAL
for ev in ["有人进入", "人离开", "心跳丢失", "复位确认", "急停", "复位确认", "人工复位"]:
    s = step(s, ev)
    print(f"  {ev:<6} → {s}")

In [ ]:
print("\n数学原理：两路独立冗余，都失效的概率相乘（编的：每路 1e-3）")
print(f"  {1e-3:.0e} × {1e-3:.0e} = {1e-3 * 1e-3:.0e}")

## 3. 代码里才看得到的

- 状态机里判断的**顺序**就是优先级：先查急停，再查 STO 要不要复位，再查故障，最后才是"有人进入"。顺序写反，急停就可能被别的分支吃掉。
- 看门狗数的是"连续"丢了几拍：中间收到一次心跳，计数就清零。

## 4. 自己改一改

- 在 `step` 里加一条：降速状态下"人继续靠近" → SS1，对照讲义第 3 步的图。
- 把看门狗的 `max_missed` 改成 2，看同一串心跳在第几拍触发。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 安全一层叠一层，越往下越简单、越不靠软件；AI 只在最上面。
> 能减速就别断电（SS1），真危险立刻断（STO）；腿足机器人先蹲下再断。